### Supplementary Figure 2

In [ ]:
import pandas as pd
from statistics import mean
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import math
from scipy.stats import mannwhitneyu

In [ ]:
### define setting to indicate raw data in saved files
database = "MGnify" # MGnify / IGC / Qin
grouping = "Groups" # Groups / Subgroups
new_discovery_study = "Werner" # ValdesMas / Werner
file_prefix = database +"_"+grouping + "_" + new_discovery_study + "_"
print(file_prefix)

In [ ]:
# Fig 1B (all identified metaproteins)
all_data = pd.read_csv("../"+file_prefix+"all_metaproteins(all_studies_normalized_5).csv", sep=",", index_col=0)#
#metaprotein_set = "all_metaproteins"
# Fig 1C (metaproteins identified in all discovery data sets)
shared_data = pd.read_csv("../"+file_prefix+"shared_metaproteins(discovery_studies_normalized_5).csv", sep=",", index_col=0)#
#metaprotein_set = "shared_metaproteins"

print(all_data.shape)
print(shared_data.shape)

### Highest abundant protein groups:

In [ ]:
###highest abundant metaprotein is extracted

#abundance_df = df_discovery
abundance_df = all_data
abundance_df.head(3)
#all_proteinsgroups_discovery
abundance_df["Average"]= 0.0
for protein_group in abundance_df.index:
    abundance_df.loc[protein_group,"Average"] = mean(abundance_df.loc[protein_group])
print(np.argmax(abundance_df["Average"]))
print(abundance_df.iloc[62])

In [ ]:
print(abundance_df["Average"].sort_values(ascending=False))
# extract average abundance and standard deviation of Chymotrypsin-like elastase family member 3B (pg: 12468)
# Average:
print("Average:"+str(abundance_df.loc[12468,"Average"]*100))
# standard deviation:
print("Standard deviation:" + str(abundance_df.loc[12468,:].drop("Average").std()*100))

### Comparision shared / not shared protein groups

In [ ]:
sharedProteinIDs = list((shared_data.index))
avg_not_shared = []
avg_shared = []
for proteinGroup in all_data.index:
#for proteinGroup in all_proteinsgroups_discovery.index:
    #print(proteinGroup)
    if proteinGroup not in sharedProteinIDs:
        avg_not_shared.append(mean(all_data.loc[proteinGroup]))
        #avg_not_shared.append(mean(all_proteinsgroups_discovery.loc[proteinGroup]))
    else:
        avg_shared.append(mean(all_data.loc[proteinGroup]))
len(avg_shared)+len(avg_not_shared) == len(all_data.index)
print(mean(avg_shared))
print(mean(avg_not_shared))

In [ ]:
# function to transform the p-values in asterics
def get_significance_asterisks(p_value):
    if p_value < 0.001:
        return 'p < 0.001'
    elif p_value < 0.01:
        return 'p < 0.01'
    elif p_value < 0.05:
        return 'p < 0.05'
    else:
        return f"p = {p_value.round(2)}"  # not significant

In [ ]:
# Plotting average abundance in logarithmic scale
fig, ax = plt.subplots()
ax.set_ylabel('Relative Abundance in [%]', fontsize=15)
box_positions=[1, 2]
new_pathway_group_abundances = [avg_shared*100, avg_not_shared*100]

bplot = ax.boxplot(new_pathway_group_abundances, positions=box_positions,
                   patch_artist=True,  # fill with color
                  )
ax.set_yscale('log')

ax.set_xticklabels(["shared protein groups", "protein groups not shared"]) #lables


sig = get_significance_asterisks(mannwhitneyu(avg_shared, avg_not_shared, method="asymptotic").pvalue)
ax.text(
    1.25, max(new_pathway_group_abundances[1])*10,                     # (x, y)-Position im Plot
    f"{sig}",         # Inhalt der Box
    fontsize=15,
    color='black',
    bbox=dict(
        facecolor='white',  # Hintergrundfarbe
        edgecolor='lightgrey',       # Rahmenfarbe
        boxstyle='round,pad=0.2', # Box-Stil
        linewidth=2
    )
)

#plt.title("Average abundance of metaproteins (shared / not shared between discovery cohorts")
plt.savefig("SupplementaryFigure2.png")
plt.show()